# Stranger Things — IMDb Rating vs Social Buzz by Season
Bar = IMDb avg rating | Line = Social Buzz Index (normalised reviews + total votes)

In [ ]:
# Install: pip install pandas matplotlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

## Step 1: Season Windows & Helper Functions

In [ ]:
SEASON_WINDOWS = [
    ("Season 1", "2016-07-15", "2017-10-26"),
    ("Season 2", "2017-10-27", "2019-07-03"),
    ("Season 3", "2019-07-04", "2022-05-26"),
    ("Season 4", "2022-05-27", "2025-11-25"),
    ("Season 5", "2025-11-26", "2099-12-31"),
]

def assign_season(date):
    for season, start, end in SEASON_WINDOWS:
        if pd.Timestamp(start) <= date <= pd.Timestamp(end):
            return season
    return "Unknown"

def norm(s):
    return (s - s.min()) / (s.max() - s.min())

## Step 2: Load & Process Data

In [ ]:
df = pd.read_csv("stranger_things_reviews.csv")

df["date"]        = pd.to_datetime(df["date"], format="mixed")
df["season"]      = df["date"].apply(assign_season)
df["helpful"]     = pd.to_numeric(df["helpful"], errors="coerce").fillna(0)
df["not_helpful"] = pd.to_numeric(df["not_helpful"], errors="coerce").fillna(0)
df["total_votes"] = df["helpful"] + df["not_helpful"]

order = ["Season 1","Season 2","Season 3","Season 4","Season 5"]

stats = (
    df.groupby("season")
    .agg(
        avg_rating=("rating", "mean"),
        n_reviews=("rating", "count"),
        avg_total_votes=("total_votes", "mean"),
    )
    .round(2)
    .reindex(order)
)

stats["norm_reviews"]     = norm(stats["n_reviews"])
stats["norm_total_votes"] = norm(stats["avg_total_votes"])
stats["social_buzz"]      = (
    0.5 * stats["norm_reviews"] + 0.5 * stats["norm_total_votes"]
).round(3)

print(stats[["avg_rating", "n_reviews", "avg_total_votes", "social_buzz"]])

## Step 3: Plot

In [ ]:
seasons = ["S1", "S2", "S3", "S4", "S5"]
x       = np.arange(len(seasons))

fig, ax1 = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#1a1a2e")
ax1.set_facecolor("#1a1a2e")

bar_colors = ["#E63946" if r == stats["avg_rating"].max()
              else "#6A5ACD" for r in stats["avg_rating"]]

bars = ax1.bar(x, stats["avg_rating"], width=0.5,
               color=bar_colors, alpha=0.85, zorder=2)

for bar, rating in zip(bars, stats["avg_rating"]):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() - 0.15,
        f"{rating:.2f}",
        ha="center", va="top",
        fontsize=12, fontweight="bold", color="white"
    )

ax1.set_ylim(6, 9)
ax1.set_ylabel("IMDb Avg Rating", color="white", fontsize=11)
ax1.tick_params(axis="y", colors="white")
ax1.set_xticks(x)
ax1.set_xticklabels(seasons, color="white", fontsize=11)
ax1.tick_params(axis="x", colors="white")
ax1.spines[:].set_visible(False)
ax1.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f"))

ax2 = ax1.twinx()
ax2.plot(x, stats["social_buzz"], color="#FFB703", linewidth=2,
         linestyle="--", marker="o", markersize=7, zorder=3)

for i, val in enumerate(stats["social_buzz"]):
    ax2.text(x[i], val + 0.02, f"{val:.2f}",
             ha="center", fontsize=9, color="#FFB703", fontweight="bold")

ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Social Buzz Index (0-1)", color="white", fontsize=11)
ax2.tick_params(axis="y", colors="white")
ax2.spines[:].set_visible(False)

legend_elements = [
    Patch(facecolor="#6A5ACD", label="IMDb avg rating"),
    Line2D([0], [0], color="#FFB703", linestyle="--",
           marker="o", markersize=6, label="Social buzz index"),
]
ax1.legend(handles=legend_elements, loc="upper right",
           facecolor="#2d2d44", edgecolor="none",
           labelcolor="white", fontsize=10)

plt.title("Stranger Things — IMDb Rating vs Social Buzz by Season",
          color="white", fontsize=15, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("rating_vs_buzz.png", dpi=150, bbox_inches="tight", facecolor="#1a1a2e")
print("Saved: rating_vs_buzz.png")
plt.show()